# FBL1N — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.vw_ds_fin_fbl1n`
**Transação SAP:** FBL1N · **Objeto:** **VIEW**
**Domínio:** Partidas individuais de fornecedor (contas a pagar)
**Filtro do cenário:** `cod_centro = '4128'`
**Colunas:** 73 · **Clustering declarado:** _(nenhum)_

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Objetivo

Extrair e caracterizar **toda** a base do centro 4128 para comparação com o extrato do SAP.

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria e representatividade |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas e timestamps |
| 13 | Códigos e zeros à esquerda |
| **14** | **Chaves normalizadas para join** |
| **15** | **Checksum de linha** |
| 16 | Amostra |
| 17 | Distribuição interna |
| **18** | **Análises específicas da transação** |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 15.


## 1. Metadados

Este objeto é uma **view**. `DESCRIBE DETAIL` e `DESCRIBE HISTORY` só funcionam em tabelas
Delta — por isso ficaram comentados, para não interromper o **Run All**.

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_ds_fin_fbl1n;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- Descomente apenas se o objeto for convertido em tabela Delta.
-- DESCRIBE DETAIL dev_procurement.corp_curated.vw_ds_fin_fbl1n;
-- DESCRIBE HISTORY dev_procurement.corp_curated.vw_ds_fin_fbl1n LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total.

In [0]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n)                            AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')                        AS linhas_do_cenario,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n), 4)           AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n)              AS valores_distintos_na_tabela;

## 3. Confirmação do filtro

Confirma que o valor `4128` existe e que não há variação de formato (espaços,
zeros à esquerda) que faça o filtro perder linhas silenciosamente.

**Se retornar mais de uma linha com formatos diferentes, o filtro está incompleto.**

In [0]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)         AS valor_bruto,
       length(CAST(`cod_centro` AS STRING)) AS comprimento,
       COUNT(*)                        AS linhas
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_centro` AS STRING)) = '4128'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'num_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_lancamento_contabil + num_item_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_lancamento_contabil`, `num_item_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil + num_item_lancamento_contabil' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `num_item_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_lancamento_contabil + num_item_lancamento_contabil`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_lancamento_contabil`, `num_item_lancamento_contabil`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY `num_lancamento_contabil`, `num_item_lancamento_contabil`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
),
dup AS (
  SELECT `num_lancamento_contabil`, `num_item_lancamento_contabil` FROM cen GROUP BY `num_lancamento_contabil`, `num_item_lancamento_contabil` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_lancamento_contabil` <=> dup.`num_lancamento_contabil` AND c.`num_item_lancamento_contabil` <=> dup.`num_item_lancamento_contabil`
),
agg AS (
  SELECT `num_lancamento_contabil`, `num_item_lancamento_contabil`,
         COUNT(DISTINCT `dt_lancamento`) AS `dt_lancamento`,
         COUNT(DISTINCT `dt_vencimento`) AS `dt_vencimento`,
         COUNT(DISTINCT `cod_exercicio_periodo`) AS `cod_exercicio_periodo`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_local_negocio`) AS `cod_local_negocio`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `cod_razao_especial`) AS `cod_razao_especial`,
         COUNT(DISTINCT `dt_vencimento_desconto_1`) AS `dt_vencimento_desconto_1`,
         COUNT(DISTINCT `cod_referencia_documento`) AS `cod_referencia_documento`,
         COUNT(DISTINCT `desc_cabecalho_lancamento_contabil`) AS `desc_cabecalho_lancamento_contabil`,
         COUNT(DISTINCT `cod_atribuicao`) AS `cod_atribuicao`,
         COUNT(DISTINCT `vl_moeda_transacao`) AS `vl_moeda_transacao`,
         COUNT(DISTINCT `cod_metodo_pagamento`) AS `cod_metodo_pagamento`,
         COUNT(DISTINCT `cod_motivo_bloqueio_pagamento`) AS `cod_motivo_bloqueio_pagamento`,
         COUNT(DISTINCT `nm_usuario_criou_lancamento_contabil`) AS `nm_usuario_criou_lancamento_contabil`,
         COUNT(DISTINCT `cod_lancamento_contabil_compensacao`) AS `cod_lancamento_contabil_compensacao`,
         COUNT(DISTINCT `num_exercicio_fiscal`) AS `num_exercicio_fiscal`,
         COUNT(DISTINCT `tp_categoria_documento_contabil`) AS `tp_categoria_documento_contabil`,
         COUNT(DISTINCT `cod_periodo_contabil`) AS `cod_periodo_contabil`,
         COUNT(DISTINCT `tp_lancamento_contabil`) AS `tp_lancamento_contabil`,
         COUNT(DISTINCT `num_documento_referencia_original`) AS `num_documento_referencia_original`,
         COUNT(DISTINCT `dt_compensacao`) AS `dt_compensacao`,
         COUNT(DISTINCT `num_exercicio_fiscal_compensacao_obsoleto`) AS `num_exercicio_fiscal_compensacao_obsoleto`,
         COUNT(DISTINCT `ind_compensacao_estornada`) AS `ind_compensacao_estornada`,
         COUNT(DISTINCT `dt_documento`) AS `dt_documento`,
         COUNT(DISTINCT `cod_referencia_parceiro_negocios_1`) AS `cod_referencia_parceiro_negocios_1`,
         COUNT(DISTINCT `cod_referencia_parceiro_negocios_2`) AS `cod_referencia_parceiro_negocios_2`,
         COUNT(DISTINCT `cod_referencia_parceiro_negocios_3`) AS `cod_referencia_parceiro_negocios_3`,
         COUNT(DISTINCT `num_documento_referencia_fatura`) AS `num_documento_referencia_fatura`,
         COUNT(DISTINCT `tp_documento_subsequente`) AS `tp_documento_subsequente`,
         COUNT(DISTINCT `cod_referencia_pagamento`) AS `cod_referencia_pagamento`,
         COUNT(DISTINCT `cod_conta_razao`) AS `cod_conta_razao`,
         COUNT(DISTINCT `cod_conta_contrapartida`) AS `cod_conta_contrapartida`,
         COUNT(DISTINCT `tp_conta`) AS `tp_conta`,
         COUNT(DISTINCT `tp_transacao_razao_especial`) AS `tp_transacao_razao_especial`,
         COUNT(DISTINCT `cod_lancamento`) AS `cod_lancamento`,
         COUNT(DISTINCT `cod_debito_credito`) AS `cod_debito_credito`,
         COUNT(DISTINCT `cod_iva`) AS `cod_iva`,
         COUNT(DISTINCT `cod_condicao_pagamento`) AS `cod_condicao_pagamento`,
         COUNT(DISTINCT `cod_motivo_diferencas_pagamento`) AS `cod_motivo_diferencas_pagamento`,
         COUNT(DISTINCT `cod_centro_lucro`) AS `cod_centro_lucro`,
         COUNT(DISTINCT `cod_divisao`) AS `cod_divisao`,
         COUNT(DISTINCT `id_sociedade_parceira`) AS `id_sociedade_parceira`,
         COUNT(DISTINCT `num_documento_compra`) AS `num_documento_compra`,
         COUNT(DISTINCT `num_item_documento_compra`) AS `num_item_documento_compra`,
         COUNT(DISTINCT `cod_elemento_pep`) AS `cod_elemento_pep`,
         COUNT(DISTINCT `cod_codigo_transacao`) AS `cod_codigo_transacao`,
         COUNT(DISTINCT `cod_lista_fatura_coletiva`) AS `cod_lista_fatura_coletiva`,
         COUNT(DISTINCT `cod_moeda_transacao`) AS `cod_moeda_transacao`,
         COUNT(DISTINCT `cod_moeda_empresa`) AS `cod_moeda_empresa`,
         COUNT(DISTINCT `cod_moeda_adicional_2`) AS `cod_moeda_adicional_2`,
         COUNT(DISTINCT `vl_base_desconto`) AS `vl_base_desconto`,
         COUNT(DISTINCT `vl_desconto`) AS `vl_desconto`,
         COUNT(DISTINCT `vl_moeda_adicional_1`) AS `vl_moeda_adicional_1`,
         COUNT(DISTINCT `vl_moeda_adicional_2`) AS `vl_moeda_adicional_2`,
         COUNT(DISTINCT `vl_moeda_interna`) AS `vl_moeda_interna`,
         COUNT(DISTINCT `desc_texto_item`) AS `desc_texto_item`,
         COUNT(DISTINCT `num_documento_estorno`) AS `num_documento_estorno`,
         COUNT(DISTINCT `tp_contrato_tesouraria`) AS `tp_contrato_tesouraria`,
         COUNT(DISTINCT `cod_contrato_imobilizado`) AS `cod_contrato_imobilizado`,
         COUNT(DISTINCT `tp_movimento_caixa`) AS `tp_movimento_caixa`,
         COUNT(DISTINCT `tp_ordem`) AS `tp_ordem`,
         COUNT(DISTINCT `desc_ordem`) AS `desc_ordem`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `cod_cnpj`) AS `cod_cnpj`,
         COUNT(DISTINCT `nm_fornecedor_nome_completo`) AS `nm_fornecedor_nome_completo`,
         COUNT(DISTINCT `dt_ingestao`) AS `dt_ingestao`
  FROM d GROUP BY `num_lancamento_contabil`, `num_item_lancamento_contabil`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(71,
    'dt_lancamento', MAX(`dt_lancamento`),
    'dt_vencimento', MAX(`dt_vencimento`),
    'cod_exercicio_periodo', MAX(`cod_exercicio_periodo`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_local_negocio', MAX(`cod_local_negocio`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'cod_razao_especial', MAX(`cod_razao_especial`),
    'dt_vencimento_desconto_1', MAX(`dt_vencimento_desconto_1`),
    'cod_referencia_documento', MAX(`cod_referencia_documento`),
    'desc_cabecalho_lancamento_contabil', MAX(`desc_cabecalho_lancamento_contabil`),
    'cod_atribuicao', MAX(`cod_atribuicao`),
    'vl_moeda_transacao', MAX(`vl_moeda_transacao`),
    'cod_metodo_pagamento', MAX(`cod_metodo_pagamento`),
    'cod_motivo_bloqueio_pagamento', MAX(`cod_motivo_bloqueio_pagamento`),
    'nm_usuario_criou_lancamento_contabil', MAX(`nm_usuario_criou_lancamento_contabil`),
    'cod_lancamento_contabil_compensacao', MAX(`cod_lancamento_contabil_compensacao`),
    'num_exercicio_fiscal', MAX(`num_exercicio_fiscal`),
    'tp_categoria_documento_contabil', MAX(`tp_categoria_documento_contabil`),
    'cod_periodo_contabil', MAX(`cod_periodo_contabil`),
    'tp_lancamento_contabil', MAX(`tp_lancamento_contabil`),
    'num_documento_referencia_original', MAX(`num_documento_referencia_original`),
    'dt_compensacao', MAX(`dt_compensacao`),
    'num_exercicio_fiscal_compensacao_obsoleto', MAX(`num_exercicio_fiscal_compensacao_obsoleto`),
    'ind_compensacao_estornada', MAX(`ind_compensacao_estornada`),
    'dt_documento', MAX(`dt_documento`),
    'cod_referencia_parceiro_negocios_1', MAX(`cod_referencia_parceiro_negocios_1`),
    'cod_referencia_parceiro_negocios_2', MAX(`cod_referencia_parceiro_negocios_2`),
    'cod_referencia_parceiro_negocios_3', MAX(`cod_referencia_parceiro_negocios_3`),
    'num_documento_referencia_fatura', MAX(`num_documento_referencia_fatura`),
    'tp_documento_subsequente', MAX(`tp_documento_subsequente`),
    'cod_referencia_pagamento', MAX(`cod_referencia_pagamento`),
    'cod_conta_razao', MAX(`cod_conta_razao`),
    'cod_conta_contrapartida', MAX(`cod_conta_contrapartida`),
    'tp_conta', MAX(`tp_conta`),
    'tp_transacao_razao_especial', MAX(`tp_transacao_razao_especial`),
    'cod_lancamento', MAX(`cod_lancamento`),
    'cod_debito_credito', MAX(`cod_debito_credito`),
    'cod_iva', MAX(`cod_iva`),
    'cod_condicao_pagamento', MAX(`cod_condicao_pagamento`),
    'cod_motivo_diferencas_pagamento', MAX(`cod_motivo_diferencas_pagamento`),
    'cod_centro_lucro', MAX(`cod_centro_lucro`),
    'cod_divisao', MAX(`cod_divisao`),
    'id_sociedade_parceira', MAX(`id_sociedade_parceira`),
    'num_documento_compra', MAX(`num_documento_compra`),
    'num_item_documento_compra', MAX(`num_item_documento_compra`),
    'cod_elemento_pep', MAX(`cod_elemento_pep`),
    'cod_codigo_transacao', MAX(`cod_codigo_transacao`),
    'cod_lista_fatura_coletiva', MAX(`cod_lista_fatura_coletiva`),
    'cod_moeda_transacao', MAX(`cod_moeda_transacao`),
    'cod_moeda_empresa', MAX(`cod_moeda_empresa`),
    'cod_moeda_adicional_2', MAX(`cod_moeda_adicional_2`),
    'vl_base_desconto', MAX(`vl_base_desconto`),
    'vl_desconto', MAX(`vl_desconto`),
    'vl_moeda_adicional_1', MAX(`vl_moeda_adicional_1`),
    'vl_moeda_adicional_2', MAX(`vl_moeda_adicional_2`),
    'vl_moeda_interna', MAX(`vl_moeda_interna`),
    'desc_texto_item', MAX(`desc_texto_item`),
    'num_documento_estorno', MAX(`num_documento_estorno`),
    'tp_contrato_tesouraria', MAX(`tp_contrato_tesouraria`),
    'cod_contrato_imobilizado', MAX(`cod_contrato_imobilizado`),
    'tp_movimento_caixa', MAX(`tp_movimento_caixa`),
    'tp_ordem', MAX(`tp_ordem`),
    'desc_ordem', MAX(`desc_ordem`),
    'cod_centro', MAX(`cod_centro`),
    'cod_material', MAX(`cod_material`),
    'desc_material', MAX(`desc_material`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'cod_cnpj', MAX(`cod_cnpj`),
    'nm_fornecedor_nome_completo', MAX(`nm_fornecedor_nome_completo`),
    'dt_ingestao', MAX(`dt_ingestao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.
Por isso a varredura é feita sobre o recorte, não sobre a base completa.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'),
perf AS (
  SELECT stack(73,
    'dt_lancamento', 'string', COUNT_IF(`dt_lancamento` IS NULL), COUNT_IF(`dt_lancamento` IS NOT NULL AND lower(trim(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_lancamento`) RLIKE '^0+([.,]0+)?$'),
    'dt_vencimento', 'string', COUNT_IF(`dt_vencimento` IS NULL), COUNT_IF(`dt_vencimento` IS NOT NULL AND lower(trim(`dt_vencimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_vencimento`) RLIKE '^0+([.,]0+)?$'),
    'cod_exercicio_periodo', 'string', COUNT_IF(`cod_exercicio_periodo` IS NULL), COUNT_IF(`cod_exercicio_periodo` IS NOT NULL AND lower(trim(`cod_exercicio_periodo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_exercicio_periodo`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_local_negocio', 'string', COUNT_IF(`cod_local_negocio` IS NULL), COUNT_IF(`cod_local_negocio` IS NOT NULL AND lower(trim(`cod_local_negocio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_local_negocio`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_razao_especial', 'string', COUNT_IF(`cod_razao_especial` IS NULL), COUNT_IF(`cod_razao_especial` IS NOT NULL AND lower(trim(`cod_razao_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_razao_especial`) RLIKE '^0+([.,]0+)?$'),
    'dt_vencimento_desconto_1', 'string', COUNT_IF(`dt_vencimento_desconto_1` IS NULL), COUNT_IF(`dt_vencimento_desconto_1` IS NOT NULL AND lower(trim(`dt_vencimento_desconto_1`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_vencimento_desconto_1`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_documento', 'string', COUNT_IF(`cod_referencia_documento` IS NULL), COUNT_IF(`cod_referencia_documento` IS NOT NULL AND lower(trim(`cod_referencia_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_documento`) RLIKE '^0+([.,]0+)?$'),
    'desc_cabecalho_lancamento_contabil', 'string', COUNT_IF(`desc_cabecalho_lancamento_contabil` IS NULL), COUNT_IF(`desc_cabecalho_lancamento_contabil` IS NOT NULL AND lower(trim(`desc_cabecalho_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_cabecalho_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'num_lancamento_contabil', 'string', COUNT_IF(`num_lancamento_contabil` IS NULL), COUNT_IF(`num_lancamento_contabil` IS NOT NULL AND lower(trim(`num_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cod_atribuicao', 'string', COUNT_IF(`cod_atribuicao` IS NULL), COUNT_IF(`cod_atribuicao` IS NOT NULL AND lower(trim(`cod_atribuicao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_atribuicao`) RLIKE '^0+([.,]0+)?$'),
    'vl_moeda_transacao', 'double', COUNT_IF(`vl_moeda_transacao` IS NULL), 0L, COUNT_IF(`vl_moeda_transacao` = 0),
    'cod_metodo_pagamento', 'string', COUNT_IF(`cod_metodo_pagamento` IS NULL), COUNT_IF(`cod_metodo_pagamento` IS NOT NULL AND lower(trim(`cod_metodo_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_metodo_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_motivo_bloqueio_pagamento', 'string', COUNT_IF(`cod_motivo_bloqueio_pagamento` IS NULL), COUNT_IF(`cod_motivo_bloqueio_pagamento` IS NOT NULL AND lower(trim(`cod_motivo_bloqueio_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_motivo_bloqueio_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_criou_lancamento_contabil', 'string', COUNT_IF(`nm_usuario_criou_lancamento_contabil` IS NULL), COUNT_IF(`nm_usuario_criou_lancamento_contabil` IS NOT NULL AND lower(trim(`nm_usuario_criou_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_criou_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cod_lancamento_contabil_compensacao', 'string', COUNT_IF(`cod_lancamento_contabil_compensacao` IS NULL), COUNT_IF(`cod_lancamento_contabil_compensacao` IS NOT NULL AND lower(trim(`cod_lancamento_contabil_compensacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_lancamento_contabil_compensacao`) RLIKE '^0+([.,]0+)?$'),
    'num_exercicio_fiscal', 'string', COUNT_IF(`num_exercicio_fiscal` IS NULL), COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND lower(trim(`num_exercicio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_exercicio_fiscal`) RLIKE '^0+([.,]0+)?$'),
    'num_item_lancamento_contabil', 'string', COUNT_IF(`num_item_lancamento_contabil` IS NULL), COUNT_IF(`num_item_lancamento_contabil` IS NOT NULL AND lower(trim(`num_item_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_categoria_documento_contabil', 'string', COUNT_IF(`tp_categoria_documento_contabil` IS NULL), COUNT_IF(`tp_categoria_documento_contabil` IS NOT NULL AND lower(trim(`tp_categoria_documento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_categoria_documento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cod_periodo_contabil', 'string', COUNT_IF(`cod_periodo_contabil` IS NULL), COUNT_IF(`cod_periodo_contabil` IS NOT NULL AND lower(trim(`cod_periodo_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_periodo_contabil`) RLIKE '^0+([.,]0+)?$'),
    'tp_lancamento_contabil', 'string', COUNT_IF(`tp_lancamento_contabil` IS NULL), COUNT_IF(`tp_lancamento_contabil` IS NOT NULL AND lower(trim(`tp_lancamento_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_lancamento_contabil`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_referencia_original', 'string', COUNT_IF(`num_documento_referencia_original` IS NULL), COUNT_IF(`num_documento_referencia_original` IS NOT NULL AND lower(trim(`num_documento_referencia_original`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_referencia_original`) RLIKE '^0+([.,]0+)?$'),
    'dt_compensacao', 'string', COUNT_IF(`dt_compensacao` IS NULL), COUNT_IF(`dt_compensacao` IS NOT NULL AND lower(trim(`dt_compensacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_compensacao`) RLIKE '^0+([.,]0+)?$'),
    'num_exercicio_fiscal_compensacao_obsoleto', 'string', COUNT_IF(`num_exercicio_fiscal_compensacao_obsoleto` IS NULL), COUNT_IF(`num_exercicio_fiscal_compensacao_obsoleto` IS NOT NULL AND lower(trim(`num_exercicio_fiscal_compensacao_obsoleto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_exercicio_fiscal_compensacao_obsoleto`) RLIKE '^0+([.,]0+)?$'),
    'ind_compensacao_estornada', 'string', COUNT_IF(`ind_compensacao_estornada` IS NULL), COUNT_IF(`ind_compensacao_estornada` IS NOT NULL AND lower(trim(`ind_compensacao_estornada`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_compensacao_estornada`) RLIKE '^0+([.,]0+)?$'),
    'dt_documento', 'string', COUNT_IF(`dt_documento` IS NULL), COUNT_IF(`dt_documento` IS NOT NULL AND lower(trim(`dt_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_documento`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_parceiro_negocios_1', 'string', COUNT_IF(`cod_referencia_parceiro_negocios_1` IS NULL), COUNT_IF(`cod_referencia_parceiro_negocios_1` IS NOT NULL AND lower(trim(`cod_referencia_parceiro_negocios_1`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_parceiro_negocios_1`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_parceiro_negocios_2', 'string', COUNT_IF(`cod_referencia_parceiro_negocios_2` IS NULL), COUNT_IF(`cod_referencia_parceiro_negocios_2` IS NOT NULL AND lower(trim(`cod_referencia_parceiro_negocios_2`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_parceiro_negocios_2`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_parceiro_negocios_3', 'string', COUNT_IF(`cod_referencia_parceiro_negocios_3` IS NULL), COUNT_IF(`cod_referencia_parceiro_negocios_3` IS NOT NULL AND lower(trim(`cod_referencia_parceiro_negocios_3`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_parceiro_negocios_3`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_referencia_fatura', 'string', COUNT_IF(`num_documento_referencia_fatura` IS NULL), COUNT_IF(`num_documento_referencia_fatura` IS NOT NULL AND lower(trim(`num_documento_referencia_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_referencia_fatura`) RLIKE '^0+([.,]0+)?$'),
    'tp_documento_subsequente', 'string', COUNT_IF(`tp_documento_subsequente` IS NULL), COUNT_IF(`tp_documento_subsequente` IS NOT NULL AND lower(trim(`tp_documento_subsequente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_documento_subsequente`) RLIKE '^0+([.,]0+)?$'),
    'cod_referencia_pagamento', 'string', COUNT_IF(`cod_referencia_pagamento` IS NULL), COUNT_IF(`cod_referencia_pagamento` IS NOT NULL AND lower(trim(`cod_referencia_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_referencia_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_conta_razao', 'string', COUNT_IF(`cod_conta_razao` IS NULL), COUNT_IF(`cod_conta_razao` IS NOT NULL AND lower(trim(`cod_conta_razao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conta_razao`) RLIKE '^0+([.,]0+)?$'),
    'cod_conta_contrapartida', 'string', COUNT_IF(`cod_conta_contrapartida` IS NULL), COUNT_IF(`cod_conta_contrapartida` IS NOT NULL AND lower(trim(`cod_conta_contrapartida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_conta_contrapartida`) RLIKE '^0+([.,]0+)?$'),
    'tp_conta', 'string', COUNT_IF(`tp_conta` IS NULL), COUNT_IF(`tp_conta` IS NOT NULL AND lower(trim(`tp_conta`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_conta`) RLIKE '^0+([.,]0+)?$'),
    'tp_transacao_razao_especial', 'string', COUNT_IF(`tp_transacao_razao_especial` IS NULL), COUNT_IF(`tp_transacao_razao_especial` IS NOT NULL AND lower(trim(`tp_transacao_razao_especial`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_transacao_razao_especial`) RLIKE '^0+([.,]0+)?$'),
    'cod_lancamento', 'string', COUNT_IF(`cod_lancamento` IS NULL), COUNT_IF(`cod_lancamento` IS NOT NULL AND lower(trim(`cod_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_lancamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_debito_credito', 'string', COUNT_IF(`cod_debito_credito` IS NULL), COUNT_IF(`cod_debito_credito` IS NOT NULL AND lower(trim(`cod_debito_credito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_debito_credito`) RLIKE '^0+([.,]0+)?$'),
    'cod_iva', 'string', COUNT_IF(`cod_iva` IS NULL), COUNT_IF(`cod_iva` IS NOT NULL AND lower(trim(`cod_iva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_iva`) RLIKE '^0+([.,]0+)?$'),
    'cod_condicao_pagamento', 'string', COUNT_IF(`cod_condicao_pagamento` IS NULL), COUNT_IF(`cod_condicao_pagamento` IS NOT NULL AND lower(trim(`cod_condicao_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_condicao_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_motivo_diferencas_pagamento', 'string', COUNT_IF(`cod_motivo_diferencas_pagamento` IS NULL), COUNT_IF(`cod_motivo_diferencas_pagamento` IS NOT NULL AND lower(trim(`cod_motivo_diferencas_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_motivo_diferencas_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_lucro', 'string', COUNT_IF(`cod_centro_lucro` IS NULL), COUNT_IF(`cod_centro_lucro` IS NOT NULL AND lower(trim(`cod_centro_lucro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_lucro`) RLIKE '^0+([.,]0+)?$'),
    'cod_divisao', 'string', COUNT_IF(`cod_divisao` IS NULL), COUNT_IF(`cod_divisao` IS NOT NULL AND lower(trim(`cod_divisao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_divisao`) RLIKE '^0+([.,]0+)?$'),
    'id_sociedade_parceira', 'string', COUNT_IF(`id_sociedade_parceira` IS NULL), COUNT_IF(`id_sociedade_parceira` IS NOT NULL AND lower(trim(`id_sociedade_parceira`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`id_sociedade_parceira`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_compra', 'string', COUNT_IF(`num_documento_compra` IS NULL), COUNT_IF(`num_documento_compra` IS NOT NULL AND lower(trim(`num_documento_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_documento_compra', 'string', COUNT_IF(`num_item_documento_compra` IS NULL), COUNT_IF(`num_item_documento_compra` IS NOT NULL AND lower(trim(`num_item_documento_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_documento_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_elemento_pep', 'string', COUNT_IF(`cod_elemento_pep` IS NULL), COUNT_IF(`cod_elemento_pep` IS NOT NULL AND lower(trim(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$'),
    'cod_codigo_transacao', 'string', COUNT_IF(`cod_codigo_transacao` IS NULL), COUNT_IF(`cod_codigo_transacao` IS NOT NULL AND lower(trim(`cod_codigo_transacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_codigo_transacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_lista_fatura_coletiva', 'string', COUNT_IF(`cod_lista_fatura_coletiva` IS NULL), COUNT_IF(`cod_lista_fatura_coletiva` IS NOT NULL AND lower(trim(`cod_lista_fatura_coletiva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_lista_fatura_coletiva`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_transacao', 'string', COUNT_IF(`cod_moeda_transacao` IS NULL), COUNT_IF(`cod_moeda_transacao` IS NOT NULL AND lower(trim(`cod_moeda_transacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_transacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_empresa', 'string', COUNT_IF(`cod_moeda_empresa` IS NULL), COUNT_IF(`cod_moeda_empresa` IS NOT NULL AND lower(trim(`cod_moeda_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_moeda_adicional_2', 'string', COUNT_IF(`cod_moeda_adicional_2` IS NULL), COUNT_IF(`cod_moeda_adicional_2` IS NOT NULL AND lower(trim(`cod_moeda_adicional_2`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_adicional_2`) RLIKE '^0+([.,]0+)?$'),
    'vl_base_desconto', 'double', COUNT_IF(`vl_base_desconto` IS NULL), 0L, COUNT_IF(`vl_base_desconto` = 0),
    'vl_desconto', 'double', COUNT_IF(`vl_desconto` IS NULL), 0L, COUNT_IF(`vl_desconto` = 0),
    'vl_moeda_adicional_1', 'double', COUNT_IF(`vl_moeda_adicional_1` IS NULL), 0L, COUNT_IF(`vl_moeda_adicional_1` = 0),
    'vl_moeda_adicional_2', 'double', COUNT_IF(`vl_moeda_adicional_2` IS NULL), 0L, COUNT_IF(`vl_moeda_adicional_2` = 0),
    'vl_moeda_interna', 'double', COUNT_IF(`vl_moeda_interna` IS NULL), 0L, COUNT_IF(`vl_moeda_interna` = 0),
    'desc_texto_item', 'string', COUNT_IF(`desc_texto_item` IS NULL), COUNT_IF(`desc_texto_item` IS NOT NULL AND lower(trim(`desc_texto_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_texto_item`) RLIKE '^0+([.,]0+)?$'),
    'num_documento_estorno', 'string', COUNT_IF(`num_documento_estorno` IS NULL), COUNT_IF(`num_documento_estorno` IS NOT NULL AND lower(trim(`num_documento_estorno`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento_estorno`) RLIKE '^0+([.,]0+)?$'),
    'tp_contrato_tesouraria', 'string', COUNT_IF(`tp_contrato_tesouraria` IS NULL), COUNT_IF(`tp_contrato_tesouraria` IS NOT NULL AND lower(trim(`tp_contrato_tesouraria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_contrato_tesouraria`) RLIKE '^0+([.,]0+)?$'),
    'cod_contrato_imobilizado', 'string', COUNT_IF(`cod_contrato_imobilizado` IS NULL), COUNT_IF(`cod_contrato_imobilizado` IS NOT NULL AND lower(trim(`cod_contrato_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_contrato_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'tp_movimento_caixa', 'string', COUNT_IF(`tp_movimento_caixa` IS NULL), COUNT_IF(`tp_movimento_caixa` IS NOT NULL AND lower(trim(`tp_movimento_caixa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_movimento_caixa`) RLIKE '^0+([.,]0+)?$'),
    'tp_ordem', 'string', COUNT_IF(`tp_ordem` IS NULL), COUNT_IF(`tp_ordem` IS NOT NULL AND lower(trim(`tp_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ordem`) RLIKE '^0+([.,]0+)?$'),
    'desc_ordem', 'string', COUNT_IF(`desc_ordem` IS NULL), COUNT_IF(`desc_ordem` IS NOT NULL AND lower(trim(`desc_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_cnpj', 'string', COUNT_IF(`cod_cnpj` IS NULL), COUNT_IF(`cod_cnpj` IS NOT NULL AND lower(trim(`cod_cnpj`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_cnpj`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor_nome_completo', 'string', COUNT_IF(`nm_fornecedor_nome_completo` IS NULL), COUNT_IF(`nm_fornecedor_nome_completo` IS NOT NULL AND lower(trim(`nm_fornecedor_nome_completo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor_nome_completo`) RLIKE '^0+([.,]0+)?$'),
    'dt_ingestao', 'date', COUNT_IF(`dt_ingestao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [0]:
-- CARDINALIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'),
card AS (
  SELECT stack(73,
    'dt_lancamento', 'string', approx_count_distinct(`dt_lancamento`),
    'dt_vencimento', 'string', approx_count_distinct(`dt_vencimento`),
    'cod_exercicio_periodo', 'string', approx_count_distinct(`cod_exercicio_periodo`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_local_negocio', 'string', approx_count_distinct(`cod_local_negocio`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'cod_razao_especial', 'string', approx_count_distinct(`cod_razao_especial`),
    'dt_vencimento_desconto_1', 'string', approx_count_distinct(`dt_vencimento_desconto_1`),
    'cod_referencia_documento', 'string', approx_count_distinct(`cod_referencia_documento`),
    'desc_cabecalho_lancamento_contabil', 'string', approx_count_distinct(`desc_cabecalho_lancamento_contabil`),
    'num_lancamento_contabil', 'string', approx_count_distinct(`num_lancamento_contabil`),
    'cod_atribuicao', 'string', approx_count_distinct(`cod_atribuicao`),
    'vl_moeda_transacao', 'double', approx_count_distinct(`vl_moeda_transacao`),
    'cod_metodo_pagamento', 'string', approx_count_distinct(`cod_metodo_pagamento`),
    'cod_motivo_bloqueio_pagamento', 'string', approx_count_distinct(`cod_motivo_bloqueio_pagamento`),
    'nm_usuario_criou_lancamento_contabil', 'string', approx_count_distinct(`nm_usuario_criou_lancamento_contabil`),
    'cod_lancamento_contabil_compensacao', 'string', approx_count_distinct(`cod_lancamento_contabil_compensacao`),
    'num_exercicio_fiscal', 'string', approx_count_distinct(`num_exercicio_fiscal`),
    'num_item_lancamento_contabil', 'string', approx_count_distinct(`num_item_lancamento_contabil`),
    'tp_categoria_documento_contabil', 'string', approx_count_distinct(`tp_categoria_documento_contabil`),
    'cod_periodo_contabil', 'string', approx_count_distinct(`cod_periodo_contabil`),
    'tp_lancamento_contabil', 'string', approx_count_distinct(`tp_lancamento_contabil`),
    'num_documento_referencia_original', 'string', approx_count_distinct(`num_documento_referencia_original`),
    'dt_compensacao', 'string', approx_count_distinct(`dt_compensacao`),
    'num_exercicio_fiscal_compensacao_obsoleto', 'string', approx_count_distinct(`num_exercicio_fiscal_compensacao_obsoleto`),
    'ind_compensacao_estornada', 'string', approx_count_distinct(`ind_compensacao_estornada`),
    'dt_documento', 'string', approx_count_distinct(`dt_documento`),
    'cod_referencia_parceiro_negocios_1', 'string', approx_count_distinct(`cod_referencia_parceiro_negocios_1`),
    'cod_referencia_parceiro_negocios_2', 'string', approx_count_distinct(`cod_referencia_parceiro_negocios_2`),
    'cod_referencia_parceiro_negocios_3', 'string', approx_count_distinct(`cod_referencia_parceiro_negocios_3`),
    'num_documento_referencia_fatura', 'string', approx_count_distinct(`num_documento_referencia_fatura`),
    'tp_documento_subsequente', 'string', approx_count_distinct(`tp_documento_subsequente`),
    'cod_referencia_pagamento', 'string', approx_count_distinct(`cod_referencia_pagamento`),
    'cod_conta_razao', 'string', approx_count_distinct(`cod_conta_razao`),
    'cod_conta_contrapartida', 'string', approx_count_distinct(`cod_conta_contrapartida`),
    'tp_conta', 'string', approx_count_distinct(`tp_conta`),
    'tp_transacao_razao_especial', 'string', approx_count_distinct(`tp_transacao_razao_especial`),
    'cod_lancamento', 'string', approx_count_distinct(`cod_lancamento`),
    'cod_debito_credito', 'string', approx_count_distinct(`cod_debito_credito`),
    'cod_iva', 'string', approx_count_distinct(`cod_iva`),
    'cod_condicao_pagamento', 'string', approx_count_distinct(`cod_condicao_pagamento`),
    'cod_motivo_diferencas_pagamento', 'string', approx_count_distinct(`cod_motivo_diferencas_pagamento`),
    'cod_centro_lucro', 'string', approx_count_distinct(`cod_centro_lucro`),
    'cod_divisao', 'string', approx_count_distinct(`cod_divisao`),
    'id_sociedade_parceira', 'string', approx_count_distinct(`id_sociedade_parceira`),
    'num_documento_compra', 'string', approx_count_distinct(`num_documento_compra`),
    'num_item_documento_compra', 'string', approx_count_distinct(`num_item_documento_compra`),
    'cod_elemento_pep', 'string', approx_count_distinct(`cod_elemento_pep`),
    'cod_codigo_transacao', 'string', approx_count_distinct(`cod_codigo_transacao`),
    'cod_lista_fatura_coletiva', 'string', approx_count_distinct(`cod_lista_fatura_coletiva`),
    'cod_moeda_transacao', 'string', approx_count_distinct(`cod_moeda_transacao`),
    'cod_moeda_empresa', 'string', approx_count_distinct(`cod_moeda_empresa`),
    'cod_moeda_adicional_2', 'string', approx_count_distinct(`cod_moeda_adicional_2`),
    'vl_base_desconto', 'double', approx_count_distinct(`vl_base_desconto`),
    'vl_desconto', 'double', approx_count_distinct(`vl_desconto`),
    'vl_moeda_adicional_1', 'double', approx_count_distinct(`vl_moeda_adicional_1`),
    'vl_moeda_adicional_2', 'double', approx_count_distinct(`vl_moeda_adicional_2`),
    'vl_moeda_interna', 'double', approx_count_distinct(`vl_moeda_interna`),
    'desc_texto_item', 'string', approx_count_distinct(`desc_texto_item`),
    'num_documento_estorno', 'string', approx_count_distinct(`num_documento_estorno`),
    'tp_contrato_tesouraria', 'string', approx_count_distinct(`tp_contrato_tesouraria`),
    'cod_contrato_imobilizado', 'string', approx_count_distinct(`cod_contrato_imobilizado`),
    'tp_movimento_caixa', 'string', approx_count_distinct(`tp_movimento_caixa`),
    'tp_ordem', 'string', approx_count_distinct(`tp_ordem`),
    'desc_ordem', 'string', approx_count_distinct(`desc_ordem`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'cod_cnpj', 'string', approx_count_distinct(`cod_cnpj`),
    'nm_fornecedor_nome_completo', 'string', approx_count_distinct(`nm_fornecedor_nome_completo`),
    'dt_ingestao', 'date', approx_count_distinct(`dt_ingestao`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma, dentro do cenário.

In [0]:
-- DOMINIO DAS CATEGORICAS NO CENARIO
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'num_exercicio_fiscal' AS coluna, CAST(`num_exercicio_fiscal` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `num_exercicio_fiscal` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_periodo_contabil' AS coluna, CAST(`cod_periodo_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_periodo_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_conta' AS coluna, CAST(`tp_conta` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `tp_conta` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_lancamento_contabil' AS coluna, CAST(`tp_lancamento_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `tp_lancamento_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_debito_credito' AS coluna, CAST(`cod_debito_credito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_debito_credito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_metodo_pagamento' AS coluna, CAST(`cod_metodo_pagamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_metodo_pagamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_motivo_bloqueio_pagamento' AS coluna, CAST(`cod_motivo_bloqueio_pagamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_motivo_bloqueio_pagamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda_transacao' AS coluna, CAST(`cod_moeda_transacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_moeda_transacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_categoria_documento_contabil' AS coluna, CAST(`tp_categoria_documento_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `tp_categoria_documento_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_razao_especial' AS coluna, CAST(`cod_razao_especial` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_razao_especial` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_condicao_pagamento' AS coluna, CAST(`cod_condicao_pagamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `cod_condicao_pagamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_compensacao_estornada' AS coluna, CAST(`ind_compensacao_estornada` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128' GROUP BY `ind_compensacao_estornada` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

A tabela tem **6 colunas `double`** entre as analisadas — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO NO CENARIO
SELECT * FROM (
  SELECT stack(6,
    'vl_moeda_transacao', 'double', COUNT(`vl_moeda_transacao`), CAST(MIN(`vl_moeda_transacao`) AS DOUBLE), CAST(MAX(`vl_moeda_transacao`) AS DOUBLE), CAST(AVG(`vl_moeda_transacao`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_transacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_transacao` < 0), COUNT_IF(`vl_moeda_transacao` = 0),
    'vl_base_desconto', 'double', COUNT(`vl_base_desconto`), CAST(MIN(`vl_base_desconto`) AS DOUBLE), CAST(MAX(`vl_base_desconto`) AS DOUBLE), CAST(AVG(`vl_base_desconto`) AS DOUBLE), CAST(percentile_approx(`vl_base_desconto`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_desconto` < 0), COUNT_IF(`vl_base_desconto` = 0),
    'vl_desconto', 'double', COUNT(`vl_desconto`), CAST(MIN(`vl_desconto`) AS DOUBLE), CAST(MAX(`vl_desconto`) AS DOUBLE), CAST(AVG(`vl_desconto`) AS DOUBLE), CAST(percentile_approx(`vl_desconto`, 0.5) AS DOUBLE), COUNT_IF(`vl_desconto` < 0), COUNT_IF(`vl_desconto` = 0),
    'vl_moeda_adicional_1', 'double', COUNT(`vl_moeda_adicional_1`), CAST(MIN(`vl_moeda_adicional_1`) AS DOUBLE), CAST(MAX(`vl_moeda_adicional_1`) AS DOUBLE), CAST(AVG(`vl_moeda_adicional_1`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_adicional_1`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_adicional_1` < 0), COUNT_IF(`vl_moeda_adicional_1` = 0),
    'vl_moeda_adicional_2', 'double', COUNT(`vl_moeda_adicional_2`), CAST(MIN(`vl_moeda_adicional_2`) AS DOUBLE), CAST(MAX(`vl_moeda_adicional_2`) AS DOUBLE), CAST(AVG(`vl_moeda_adicional_2`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_adicional_2`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_adicional_2` < 0), COUNT_IF(`vl_moeda_adicional_2` = 0),
    'vl_moeda_interna', 'double', COUNT(`vl_moeda_interna`), CAST(MIN(`vl_moeda_interna`) AS DOUBLE), CAST(MAX(`vl_moeda_interna`) AS DOUBLE), CAST(AVG(`vl_moeda_interna`) AS DOUBLE), CAST(percentile_approx(`vl_moeda_interna`, 0.5) AS DOUBLE), COUNT_IF(`vl_moeda_interna` < 0), COUNT_IF(`vl_moeda_interna` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(6,
    'vl_moeda_transacao', CAST(SUM(`vl_moeda_transacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_transacao`), 2) AS STRING), COUNT(`vl_moeda_transacao`),
    'vl_base_desconto', CAST(SUM(`vl_base_desconto`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_desconto`), 2) AS STRING), COUNT(`vl_base_desconto`),
    'vl_desconto', CAST(SUM(`vl_desconto`) AS DOUBLE), CAST(ROUND(SUM(`vl_desconto`), 2) AS STRING), COUNT(`vl_desconto`),
    'vl_moeda_adicional_1', CAST(SUM(`vl_moeda_adicional_1`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_adicional_1`), 2) AS STRING), COUNT(`vl_moeda_adicional_1`),
    'vl_moeda_adicional_2', CAST(SUM(`vl_moeda_adicional_2`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_adicional_2`), 2) AS STRING), COUNT(`vl_moeda_adicional_2`),
    'vl_moeda_interna', CAST(SUM(`vl_moeda_interna`) AS DOUBLE), CAST(ROUND(SUM(`vl_moeda_interna`), 2) AS STRING), COUNT(`vl_moeda_interna`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 12. Datas armazenadas como STRING

**Armadilha:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
Mesma data, formato diferente — já gerou 16.773 falsos positivos em validação anterior.

In [0]:
-- DATAS EM STRING
SELECT coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo,
       CASE WHEN (CASE WHEN fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato' ELSE 'formato unico' END AS veredito
FROM (
  SELECT stack(5,
    'dt_lancamento', COUNT_IF(`dt_lancamento` IS NULL OR trim(`dt_lancamento`) = ''), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END), MAX(CASE WHEN trim(`dt_lancamento`) NOT IN ('', '00000000') THEN `dt_lancamento` END),
    'dt_vencimento', COUNT_IF(`dt_vencimento` IS NULL OR trim(`dt_vencimento`) = ''), COUNT_IF(trim(`dt_vencimento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_vencimento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_vencimento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_vencimento`) NOT IN ('', '00000000') THEN `dt_vencimento` END), MAX(CASE WHEN trim(`dt_vencimento`) NOT IN ('', '00000000') THEN `dt_vencimento` END),
    'dt_vencimento_desconto_1', COUNT_IF(`dt_vencimento_desconto_1` IS NULL OR trim(`dt_vencimento_desconto_1`) = ''), COUNT_IF(trim(`dt_vencimento_desconto_1`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_vencimento_desconto_1`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_vencimento_desconto_1`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_vencimento_desconto_1`) NOT IN ('', '00000000') THEN `dt_vencimento_desconto_1` END), MAX(CASE WHEN trim(`dt_vencimento_desconto_1`) NOT IN ('', '00000000') THEN `dt_vencimento_desconto_1` END),
    'dt_compensacao', COUNT_IF(`dt_compensacao` IS NULL OR trim(`dt_compensacao`) = ''), COUNT_IF(trim(`dt_compensacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_compensacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_compensacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_compensacao`) NOT IN ('', '00000000') THEN `dt_compensacao` END), MAX(CASE WHEN trim(`dt_compensacao`) NOT IN ('', '00000000') THEN `dt_compensacao` END),
    'dt_documento', COUNT_IF(`dt_documento` IS NULL OR trim(`dt_documento`) = ''), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_documento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), MIN(CASE WHEN trim(`dt_documento`) NOT IN ('', '00000000') THEN `dt_documento` END), MAX(CASE WHEN trim(`dt_documento`) NOT IN ('', '00000000') THEN `dt_documento` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, minimo, maximo)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 12.1 Datas em tipo nativo

In [0]:
-- DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(1,
    'dt_ingestao', COUNT_IF(`dt_ingestao` IS NULL), CAST(MIN(`dt_ingestao`) AS STRING), CAST(MAX(`dt_ingestao`) AS STRING), COUNT(DISTINCT `dt_ingestao`), COUNT_IF(`dt_ingestao` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(11,
    'num_lancamento_contabil', 'string', COUNT_IF(CAST(`num_lancamento_contabil` AS STRING) IS NULL OR trim(CAST(`num_lancamento_contabil` AS STRING)) = ''), MIN(length(trim(CAST(`num_lancamento_contabil` AS STRING)))), MAX(length(trim(CAST(`num_lancamento_contabil` AS STRING)))), COUNT_IF(trim(CAST(`num_lancamento_contabil` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_lancamento_contabil` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_lancamento_contabil` AS STRING)), '^0+', '')),
    'num_item_lancamento_contabil', 'string', COUNT_IF(CAST(`num_item_lancamento_contabil` AS STRING) IS NULL OR trim(CAST(`num_item_lancamento_contabil` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_lancamento_contabil` AS STRING)))), MAX(length(trim(CAST(`num_item_lancamento_contabil` AS STRING)))), COUNT_IF(trim(CAST(`num_item_lancamento_contabil` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_lancamento_contabil` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_lancamento_contabil` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'cod_conta_razao', 'string', COUNT_IF(CAST(`cod_conta_razao` AS STRING) IS NULL OR trim(CAST(`cod_conta_razao` AS STRING)) = ''), MIN(length(trim(CAST(`cod_conta_razao` AS STRING)))), MAX(length(trim(CAST(`cod_conta_razao` AS STRING)))), COUNT_IF(trim(CAST(`cod_conta_razao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_conta_razao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_conta_razao` AS STRING)), '^0+', '')),
    'num_documento_compra', 'string', COUNT_IF(CAST(`num_documento_compra` AS STRING) IS NULL OR trim(CAST(`num_documento_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_documento_compra` AS STRING)))), MAX(length(trim(CAST(`num_documento_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_documento_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_documento_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_documento_compra` AS STRING)), '^0+', '')),
    'num_item_documento_compra', 'string', COUNT_IF(CAST(`num_item_documento_compra` AS STRING) IS NULL OR trim(CAST(`num_item_documento_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_documento_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_documento_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_documento_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_documento_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_documento_compra` AS STRING)), '^0+', '')),
    'cod_cnpj', 'string', COUNT_IF(CAST(`cod_cnpj` AS STRING) IS NULL OR trim(CAST(`cod_cnpj` AS STRING)) = ''), MIN(length(trim(CAST(`cod_cnpj` AS STRING)))), MAX(length(trim(CAST(`cod_cnpj` AS STRING)))), COUNT_IF(trim(CAST(`cod_cnpj` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_cnpj` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_cnpj` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 14. Chaves normalizadas para join com o SAP

Lista das chaves já **sem zeros à esquerda**, prontas para colar no Excel
e cruzar com o extrato do SAP via PROCV/ÍNDICE.

In [0]:
-- 14. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', '') AS `cod_empresa_norm`,
       regexp_replace(trim(CAST(`num_exercicio_fiscal` AS STRING)), '^0+', '') AS `num_exercicio_fiscal_norm`,
       regexp_replace(trim(CAST(`num_lancamento_contabil` AS STRING)), '^0+', '') AS `num_lancamento_contabil_norm`,
       regexp_replace(trim(CAST(`num_item_lancamento_contabil` AS STRING)), '^0+', '') AS `num_item_lancamento_contabil_norm`
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
ORDER BY 1, 2, 3, 4;

## 15. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento` AS STRING), ''), COALESCE(CAST(`cod_exercicio_periodo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_local_negocio` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_razao_especial` AS STRING), ''), COALESCE(CAST(`dt_vencimento_desconto_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`desc_cabecalho_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_atribuicao` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_metodo_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_bloqueio_pagamento` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_lancamento_contabil_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`num_item_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento_contabil` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_original` AS STRING), ''), COALESCE(CAST(`dt_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal_compensacao_obsoleto` AS STRING), ''), COALESCE(CAST(`ind_compensacao_estornada` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_2` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_3` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_fatura` AS STRING), ''), COALESCE(CAST(`tp_documento_subsequente` AS STRING), ''), COALESCE(CAST(`cod_referencia_pagamento` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_transacao_razao_especial` AS STRING), ''), COALESCE(CAST(`cod_lancamento` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_iva` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferencas_pagamento` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`id_sociedade_parceira` AS STRING), ''), COALESCE(CAST(`num_documento_compra` AS STRING), ''), COALESCE(CAST(`num_item_documento_compra` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_codigo_transacao` AS STRING), ''), COALESCE(CAST(`cod_lista_fatura_coletiva` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_base_desconto` AS STRING), ''), COALESCE(CAST(`vl_desconto` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_1` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_moeda_interna` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`num_documento_estorno` AS STRING), ''), COALESCE(CAST(`tp_contrato_tesouraria` AS STRING), ''), COALESCE(CAST(`cod_contrato_imobilizado` AS STRING), ''), COALESCE(CAST(`tp_movimento_caixa` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`desc_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_cnpj` AS STRING), ''), COALESCE(CAST(`nm_fornecedor_nome_completo` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento` AS STRING), ''), COALESCE(CAST(`cod_exercicio_periodo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_local_negocio` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_razao_especial` AS STRING), ''), COALESCE(CAST(`dt_vencimento_desconto_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`desc_cabecalho_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_atribuicao` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_metodo_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_bloqueio_pagamento` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_lancamento_contabil_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`num_item_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento_contabil` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_original` AS STRING), ''), COALESCE(CAST(`dt_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal_compensacao_obsoleto` AS STRING), ''), COALESCE(CAST(`ind_compensacao_estornada` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_2` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_3` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_fatura` AS STRING), ''), COALESCE(CAST(`tp_documento_subsequente` AS STRING), ''), COALESCE(CAST(`cod_referencia_pagamento` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_transacao_razao_especial` AS STRING), ''), COALESCE(CAST(`cod_lancamento` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_iva` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferencas_pagamento` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`id_sociedade_parceira` AS STRING), ''), COALESCE(CAST(`num_documento_compra` AS STRING), ''), COALESCE(CAST(`num_item_documento_compra` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_codigo_transacao` AS STRING), ''), COALESCE(CAST(`cod_lista_fatura_coletiva` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_base_desconto` AS STRING), ''), COALESCE(CAST(`vl_desconto` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_1` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_moeda_interna` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`num_documento_estorno` AS STRING), ''), COALESCE(CAST(`tp_contrato_tesouraria` AS STRING), ''), COALESCE(CAST(`cod_contrato_imobilizado` AS STRING), ''), COALESCE(CAST(`tp_movimento_caixa` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`desc_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_cnpj` AS STRING), ''), COALESCE(CAST(`nm_fornecedor_nome_completo` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento` AS STRING), ''), COALESCE(CAST(`cod_exercicio_periodo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_local_negocio` AS STRING), ''), COALESCE(CAST(`cod_centro_custo` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_razao_especial` AS STRING), ''), COALESCE(CAST(`dt_vencimento_desconto_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_documento` AS STRING), ''), COALESCE(CAST(`desc_cabecalho_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_atribuicao` AS STRING), ''), COALESCE(CAST(`vl_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_metodo_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_bloqueio_pagamento` AS STRING), ''), COALESCE(CAST(`nm_usuario_criou_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`cod_lancamento_contabil_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal` AS STRING), ''), COALESCE(CAST(`num_item_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`tp_categoria_documento_contabil` AS STRING), ''), COALESCE(CAST(`cod_periodo_contabil` AS STRING), ''), COALESCE(CAST(`tp_lancamento_contabil` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_original` AS STRING), ''), COALESCE(CAST(`dt_compensacao` AS STRING), ''), COALESCE(CAST(`num_exercicio_fiscal_compensacao_obsoleto` AS STRING), ''), COALESCE(CAST(`ind_compensacao_estornada` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_1` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_2` AS STRING), ''), COALESCE(CAST(`cod_referencia_parceiro_negocios_3` AS STRING), ''), COALESCE(CAST(`num_documento_referencia_fatura` AS STRING), ''), COALESCE(CAST(`tp_documento_subsequente` AS STRING), ''), COALESCE(CAST(`cod_referencia_pagamento` AS STRING), ''), COALESCE(CAST(`cod_conta_razao` AS STRING), ''), COALESCE(CAST(`cod_conta_contrapartida` AS STRING), ''), COALESCE(CAST(`tp_conta` AS STRING), ''), COALESCE(CAST(`tp_transacao_razao_especial` AS STRING), ''), COALESCE(CAST(`cod_lancamento` AS STRING), ''), COALESCE(CAST(`cod_debito_credito` AS STRING), ''), COALESCE(CAST(`cod_iva` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`cod_motivo_diferencas_pagamento` AS STRING), ''), COALESCE(CAST(`cod_centro_lucro` AS STRING), ''), COALESCE(CAST(`cod_divisao` AS STRING), ''), COALESCE(CAST(`id_sociedade_parceira` AS STRING), ''), COALESCE(CAST(`num_documento_compra` AS STRING), ''), COALESCE(CAST(`num_item_documento_compra` AS STRING), ''), COALESCE(CAST(`cod_elemento_pep` AS STRING), ''), COALESCE(CAST(`cod_codigo_transacao` AS STRING), ''), COALESCE(CAST(`cod_lista_fatura_coletiva` AS STRING), ''), COALESCE(CAST(`cod_moeda_transacao` AS STRING), ''), COALESCE(CAST(`cod_moeda_empresa` AS STRING), ''), COALESCE(CAST(`cod_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_base_desconto` AS STRING), ''), COALESCE(CAST(`vl_desconto` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_1` AS STRING), ''), COALESCE(CAST(`vl_moeda_adicional_2` AS STRING), ''), COALESCE(CAST(`vl_moeda_interna` AS STRING), ''), COALESCE(CAST(`desc_texto_item` AS STRING), ''), COALESCE(CAST(`num_documento_estorno` AS STRING), ''), COALESCE(CAST(`tp_contrato_tesouraria` AS STRING), ''), COALESCE(CAST(`cod_contrato_imobilizado` AS STRING), ''), COALESCE(CAST(`tp_movimento_caixa` AS STRING), ''), COALESCE(CAST(`tp_ordem` AS STRING), ''), COALESCE(CAST(`desc_ordem` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_material` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_cnpj` AS STRING), ''), COALESCE(CAST(`nm_fornecedor_nome_completo` AS STRING), ''), COALESCE(CAST(`dt_ingestao` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128';

## 16. Amostra de linhas completas

In [0]:
-- 16. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
ORDER BY `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `num_item_lancamento_contabil`
LIMIT 20;

## 17. Distribuição interna do centro 4128

Como o volume se reparte dentro do cenário. Útil para conferir se o extrato do SAP
tem a mesma composição.

In [0]:
-- 17. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR cod_periodo_contabil
SELECT COALESCE(NULLIF(trim(CAST(`cod_periodo_contabil` AS STRING)), ''), '(vazio)') AS `cod_periodo_contabil`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_periodo_contabil` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR tp_lancamento_contabil
SELECT COALESCE(NULLIF(trim(CAST(`tp_lancamento_contabil` AS STRING)), ''), '(vazio)') AS `tp_lancamento_contabil`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_lancamento_contabil` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR cod_metodo_pagamento
SELECT COALESCE(NULLIF(trim(CAST(`cod_metodo_pagamento` AS STRING)), ''), '(vazio)') AS `cod_metodo_pagamento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_metodo_pagamento` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 17. DISTRIBUICAO POR cod_fornecedor
SELECT COALESCE(NULLIF(trim(CAST(`cod_fornecedor` AS STRING)), ''), '(vazio)') AS `cod_fornecedor`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_fornecedor` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

### 17.1 Freshness

In [0]:
-- 17.1 FRESHNESS
SELECT `dt_ingestao` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'
GROUP BY `dt_ingestao`
ORDER BY data_carga DESC
LIMIT 30;

## 18. Análises específicas — FBL1N

### 18.1 Pendência declarada em `vl_moeda_interna`

O comentário do schema registra uma **pendência em aberto**: o dicionário aponta `DMSHB`
(moeda da empresa), mas a fonte atual seria a moeda da transação.

In [0]:
-- 18.1 vl_moeda_interna vs vl_moeda_transacao
SELECT COUNT(*)                                           AS linhas,
       COUNT_IF(vl_moeda_interna = vl_moeda_transacao)    AS valores_identicos,
       COUNT_IF(vl_moeda_interna <> vl_moeda_transacao)   AS valores_diferentes,
       ROUND(SUM(vl_moeda_interna), 2)                    AS soma_interna,
       ROUND(SUM(vl_moeda_transacao), 2)                  AS soma_transacao,
       CASE WHEN COUNT_IF(vl_moeda_interna <> vl_moeda_transacao) = 0
            THEN 'PENDENCIA CONFIRMADA - colunas identicas'
            ELSE 'colunas divergem - investigar' END      AS veredito
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128';

### 18.2 Ciclo de compensação

In [0]:
-- 18.2 CICLO DE COMPENSACAO
SELECT COUNT(*)                                                              AS linhas,
       COUNT_IF(trim(COALESCE(dt_compensacao,'')) NOT IN ('', '00000000'))   AS com_data_compensacao,
       COUNT_IF(trim(COALESCE(cod_lancamento_contabil_compensacao,'')) <> '') AS com_doc_compensacao,
       COUNT_IF(trim(COALESCE(dt_compensacao,'')) IN ('', '00000000')
            AND trim(COALESCE(cod_lancamento_contabil_compensacao,'')) <> '') AS doc_sem_data,
       COUNT_IF(trim(COALESCE(dt_compensacao,'')) NOT IN ('', '00000000')
            AND trim(COALESCE(cod_lancamento_contabil_compensacao,'')) = '')  AS data_sem_doc,
       COUNT_IF(trim(COALESCE(ind_compensacao_estornada,'')) <> '')           AS compensacao_estornada
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128';

### 18.3 Vínculo com documento de compras e fornecedores do centro

In [0]:
-- 18.3 VINCULOS
SELECT COUNT(*)                                                    AS linhas,
       COUNT_IF(trim(COALESCE(num_documento_compra,'')) <> '')     AS com_documento_compra,
       COUNT_IF(trim(COALESCE(cod_material,'')) <> '')             AS com_material,
       COUNT_IF(trim(COALESCE(cod_centro_custo,'')) <> '')         AS com_centro_custo,
       COUNT_IF(trim(COALESCE(cod_elemento_pep,'')) <> '')         AS com_elemento_pep,
       COUNT(DISTINCT cod_fornecedor)                              AS fornecedores_distintos,
       COUNT(DISTINCT cod_empresa)                                 AS empresas_distintas
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128';

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for muito grande, use a célula 19.1.

In [0]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n
WHERE `cod_centro` = '4128'
ORDER BY `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `num_item_lancamento_contabil`;

### 19.1 Alternativa para volume grande _(opcional)_

In [0]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_fbl1n_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_fbl1n_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'num_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_lancamento_contabil + num_item_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_lancamento_contabil`, `num_item_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'cod_empresa + num_exercicio_fiscal + num_lancamento_contabil + num_item_lancamento_contabil' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_empresa`, `num_exercicio_fiscal`, `num_lancamento_contabil`, `num_item_lancamento_contabil` FROM dev_procurement.corp_curated.vw_ds_fin_fbl1n WHERE `cod_centro` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'FBL1N' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'vw_ds_fin_fbl1n'
UNION ALL SELECT 'CENARIO', 'tipo de objeto', 'VIEW'
UNION ALL SELECT 'CENARIO', 'coluna de recorte', 'cod_centro'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '73'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a FBL1N no SAP com o filtro equivalente ao centro 4128.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Formato de data normalizado (seção 12)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
